In [28]:
# ZONE 1: ENVIRONMENT SETUP & IMPORTS
# ==========================================
# RULE: Run this cell FIRST before executing any other zones.

# 1. Install all required packages (UI, Visualization, Data, Architecture)
!pip install streamlit plotly numpy pandas pyngrok influxdb-client paho-mqtt pyzmq

# 2. Import standard libraries
import time
import random
import json

# 3. Import data manipulation and UI/Visualization libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# 4. Import architecture-specific libraries
import paho.mqtt.client as mqtt
import zmq
from influxdb_client import InfluxDBClient, Point, WriteOptions
from pyngrok import ngrok # For tunneling Streamlit out of Colab

In [29]:
# ZONE 2: GLOBAL STATE REGISTRY
# ==========================================
# RULES:
# 1. Do NOT create loose variables for these metrics.
# 2. Update this dictionary directly so all modules stay synced.

digital_twin_state = {
    "current_bias_voltage": 0.0,      # Vb
    "photocurrent": 0.0,              # Ip
    "dark_current": 0.0,              # Id
    "optical_power_estimate": 0.0,
    "noise_spectral_density": 0.0,
    "junction_temperature": 25.0,     # Tj (Starting at room temp)
    "bandwidth_mode": "Standard",
    "saturation_indicator": False,    # Safe status flag
    "degradation_tracking": 0.0,
    "communication_latency": 0.0
}

In [30]:
# ZONE 3: MOCK SENSOR DATA GENERATOR (TESTING)
# ==========================================
# RULE: Use this function to simulate hardware feedback when the physical photodetector is disconnected.

def generate_mock_sensor_data(current_bias):
    """
    Simulates the physical hardware responding to a given bias voltage.
    Adds slight random noise to mimic thermal drift and dark current spikes.
    """
    import random

    # Simulate a baseline temperature with slight thermal fluctuations
    mock_temp = 25.0 + random.uniform(-0.5, 1.2)

    # Simulate dark current increasing slightly as temperature rises
    mock_dark_current = 0.05 * (mock_temp / 25.0) + random.uniform(0.001, 0.005)

    # Simulate photocurrent based on the applied bias (with mock noise)
    mock_photocurrent = (current_bias * 1.5) + random.uniform(-0.1, 0.1)

    # Randomly simulate a saturation risk if bias is pushed too high
    mock_saturation = True if current_bias > 15.0 else False

    return {
        "temperature": round(mock_temp, 2),
        "dark_current": round(mock_dark_current, 4),
        "photocurrent": round(mock_photocurrent, 2),
        "is_saturated": mock_saturation
    }

# Quick Test -Run this cell to ensure the generator works before passing
print("Testing Mock Hardware:", generate_mock_sensor_data(current_bias=5.0))

Testing Mock Hardware: {'temperature': 24.63, 'dark_current': 0.0514, 'photocurrent': 7.53, 'is_saturated': False}


In [31]:
%%writefile app.py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st

st.set_page_config(page_title="Photodetector Digital Twin", layout="wide")

st.title("🔬 Photodetector Digital Twin Dashboard")
st.caption("Temporal & System State Visualization — Grounded in Optoelectronic Physics")

# Sidebar Controls

st.sidebar.header("⚡ Simulation Controls")

simulation_time = st.sidebar.slider("Simulation Duration (ms)", 100, 1000, 500, step=50)
optical_power = st.sidebar.slider("Optical Input Power (mW)", 0.1, 10.0, 3.5, step=0.1)
ambient_temp = st.sidebar.slider("Ambient Temperature (°C)", 15.0, 60.0, 25.0, step=0.5)
initial_bias = st.sidebar.slider("Initial Bias (V)", 0.5, 5.0, 2.5, step=0.1)

auto_bias = st.sidebar.checkbox("Enable AI Closed-Loop Bias Control", value=True)

# Physics Engine & State Calculation

t = np.linspace(0, simulation_time, 200)

temp_drift = ambient_temp + 0.01 * t + np.sin(t / 20) * 0.5
dark_current = 1e-9 * np.exp(0.08 * (temp_drift - 25))
responsivity = 0.85 * (1 - 0.001 * (temp_drift - 25))
photocurrent = responsivity * optical_power

if auto_bias:
    adjusted_bias = initial_bias * (1 - 0.002 * (temp_drift - 25))
else:
    adjusted_bias = np.full_like(t, initial_bias)

signal_power = photocurrent**2
noise_power = (dark_current + 1e-6) ** 2
snr_db = 10 * np.log10(signal_power / noise_power)

# Digital Twin Health Metrics

m1, m2, m3, m4 = st.columns(4)
m1.metric("Mean Photocurrent", f"{np.mean(photocurrent):.3f} mA")
m2.metric("Peak Dark Current", f"{np.max(dark_current)*1e6:.2f} µA")
m3.metric("Avg Junction Temp", f"{np.mean(temp_drift):.1f} °C")
m4.metric("Avg SNR", f"{np.mean(snr_db):.1f} dB")

st.markdown("---")

# Temporal Visualization Plots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Photocurrent & Dark Current Over Time",
        "Thermal Drift (Junction Temp)",
        "Closed-Loop Adaptive Bias Voltage",
        "Signal-to-Noise Ratio (SNR)"
    )
)

fig.add_trace(go.Scatter(x=t, y=photocurrent, mode="lines", name="Photocurrent (mA)", line=dict(color="#00CC96")), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=dark_current*1e3, mode="lines", name="Dark Current (µA x10³)", line=dict(color="#EF553B", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=temp_drift, mode="lines", name="Temp (°C)", line=dict(color="#FFA15A")), row=1, col=2)
fig.add_trace(go.Scatter(x=t, y=adjusted_bias, mode="lines", name="Bias (V)", line=dict(color="#636EFA")), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=snr_db, mode="lines", name="SNR (dB)", line=dict(color="#AB63FA")), row=2, col=2)

fig.update_layout(height=600, showlegend=True, title_text="Live Telemetry Projections")
st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [32]:
from google.colab import output
import subprocess
import time

# 1. Kill any existing instances
!pkill -f streamlit

# 2. Run Streamlit with CORS & XSRF disabled for Colab proxy compatibility
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# 3. Wait for it to start
time.sleep(3)

# 4. Generate clickable window link
print("🎉 SUCCESS! Click the link below to launch your app:\n")
output.serve_kernel_port_as_window(8501)

🎉 SUCCESS! Click the link below to launch your app:

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [33]:
# ZONE 4: AI DECISION ENGINE & BEHAVIOURAL MODEL
# =======================================================
# Objective

# The purpose of this module is to demonstrate an intelligent behavioral model for the Photodetector Digital Twin.

# The AI continuously monitors the Digital Twin state using sensor data such as:

# - Junction Temperature
# - Dark Current
# - Photocurrent
# - Saturation Indicator

## Based on these operating conditions, the AI automatically adjusts the photodetector bias voltage to maintain safe and efficient operation.

## This demonstrates a closed-loop Digital Twin where incoming sensor data directly influences the Digital Twin state.

In [34]:
def ai_bias_controller():
    """
    AI Behavioural Model
    Reads the current Digital Twin state and automatically
    adjusts the bias voltage to maintain safe operation.
    """

    # Read values from the Digital Twin state
    current_temp = digital_twin_state["junction_temperature"]
    current_bias = digital_twin_state["current_bias_voltage"]
    current_dark = digital_twin_state["dark_current"]
    current_photo = digital_twin_state["photocurrent"]
    saturation = digital_twin_state["saturation_indicator"]

    decision = "No adjustment required"

    # ---------- AI Decision Rules ----------

    # Rule 1: Device overheating
    if current_temp > 45:
        current_bias -= 0.20
        decision = "Temperature too high → Reduce bias voltage"

    # Rule 2: High dark current
    elif current_dark > 0.10:
        current_bias -= 0.10
        decision = "Dark current high → Reduce bias voltage"

    # Rule 3: Saturation detected
    elif saturation:
        current_bias -= 0.30
        decision = "Photodetector saturated → Emergency bias reduction"

    # Rule 4: Weak photocurrent
    elif current_photo < 2.0:
        current_bias += 0.10
        decision = "Weak photocurrent → Increase bias voltage"

    # Prevent invalid bias values
    current_bias = max(0.0, min(current_bias, 5.0))

    # Update Digital Twin State
    digital_twin_state["current_bias_voltage"] = current_bias

    return decision

In [38]:
# ==========================================
# AI DEMONSTRATION
# ==========================================

def run_test_case(title, temperature, dark_current, photocurrent, saturation, bias):

    digital_twin_state["junction_temperature"] = temperature
    digital_twin_state["dark_current"] = dark_current
    digital_twin_state["photocurrent"] = photocurrent
    digital_twin_state["saturation_indicator"] = saturation
    digital_twin_state["current_bias_voltage"] = bias

    print("=" * 60)
    print(title)
    print("=" * 60)

    print("Before AI Decision")
    print(f"Junction Temperature : {temperature} °C")
    print(f"Dark Current         : {dark_current:.2f} mA")
    print(f"Photocurrent         : {photocurrent:.2f} mA")
    print(f"Bias Voltage         : {bias:.2f} V")
    print(f"Saturation           : {saturation}")

    decision = ai_bias_controller()

    print("\nAI Decision")
    print(decision)

    print("\nAfter AI Decision")
    print(f"Updated Bias Voltage : {digital_twin_state['current_bias_voltage']:.2f} V")

    print("\n")


# ===============================
# Test Case 1 - Overheating Condition
# ===============================
run_test_case(
    "TEST CASE 1 : Overheating Condition",
    temperature=48,
    dark_current=0.05,
    photocurrent=2.50,
    saturation=False,
    bias=2.50
)


# ===============================
# Test Case 2 - Excessive Dark Current
# ===============================
run_test_case(
    "TEST CASE 2 : Excessive Dark Current",
    temperature=30,
    dark_current=0.15,
    photocurrent=2.50,
    saturation=False,
    bias=2.50
)


# ===============================
# Test Case 3 - Weak Optical Signal
# ===============================
run_test_case(
    "TEST CASE 3 : Weak Optical Signal",
    temperature=28,
    dark_current=0.05,
    photocurrent=1.50,
    saturation=False,
    bias=2.50
)

TEST CASE 1 : Overheating Condition
Before AI Decision
Junction Temperature : 48 °C
Dark Current         : 0.05 mA
Photocurrent         : 2.50 mA
Bias Voltage         : 2.50 V
Saturation           : False

AI Decision
Temperature too high → Reduce bias voltage

After AI Decision
Updated Bias Voltage : 2.30 V


TEST CASE 2 : Excessive Dark Current
Before AI Decision
Junction Temperature : 30 °C
Dark Current         : 0.15 mA
Photocurrent         : 2.50 mA
Bias Voltage         : 2.50 V
Saturation           : False

AI Decision
Dark current high → Reduce bias voltage

After AI Decision
Updated Bias Voltage : 2.40 V


TEST CASE 3 : Weak Optical Signal
Before AI Decision
Junction Temperature : 28 °C
Dark Current         : 0.05 mA
Photocurrent         : 1.50 mA
Bias Voltage         : 2.50 V
Saturation           : False

AI Decision
Weak photocurrent → Increase bias voltage

After AI Decision
Updated Bias Voltage : 2.60 V




In [39]:
print("=" * 60)
print("OVERALL AI MODULE SUMMARY")
print("=" * 60)

print("""
✓ Test Case 1:
  Overheating detected.
  AI reduced the bias voltage.

✓ Test Case 2:
  Excessive dark current detected.
  AI reduced the bias voltage.

✓ Test Case 3:
  Weak photocurrent detected.
  AI increased the bias voltage.

Conclusion:
The AI behavioural model successfully responded to different
photodetector operating conditions and updated the Digital
Twin state automatically.
""")

OVERALL AI MODULE SUMMARY

✓ Test Case 1:
  Overheating detected.
  AI reduced the bias voltage.

✓ Test Case 2:
  Excessive dark current detected.
  AI reduced the bias voltage.

✓ Test Case 3:
  Weak photocurrent detected.
  AI increased the bias voltage.

Conclusion:
The AI behavioural model successfully responded to different
photodetector operating conditions and updated the Digital
Twin state automatically.

